In [ ]:
# %% [markdown]
# # Algorithm Performance Comparison: A* vs DHPA*
# This notebook calculates the percentage differences between A* and DHPA* across different maps (`maze` and `mp2`).
#
# ### Definitions:
# - **A***: Search Phase Time = `found_overall_path`
# - **DHPA***: Search Phase Time = `finish_abstract_graph` + `found_graph_path` + `found_overall_path`

# %%
import json
import pandas as pd
import numpy as np

# Files to process
files = {
    'maze': 'maze.json',
    'mp2': 'mp2.json',
    'mp4': 'mp4.json',
}

all_data = []

for map_name, file_path in files.items():
    with open(file_path, 'r') as f:
        content = json.load(f)
        
    for entry in content['data']:
        algo = entry['algorithm']
        timings = entry['timings']
        
        # Compute Search Phase Time based on algorithm components
        if algo == 'a':
            search_time = timings.get('found_overall_path', 0)
        elif algo == 'dhpa':
            search_time = (
                timings.get('finish_abstract_graph', 0) +
                timings.get('found_graph_path', 0) +
                timings.get('found_overall_path', 0)
            )
        
        max_memory = timings.get('max_memory', 0)
        path_length = timings.get('path_length', 0)
        
        all_data.append({
            'map': map_name,
            'algorithm': algo,
            'search_time': search_time,
            'max_memory': max_memory,
            'path_length': path_length
        })

df = pd.DataFrame(all_data)
print("Data loaded successfully! Total runs found:", len(df))

# %% [markdown]
# ### Aggregate Mean Values Across Runs

# %%
# Group by map and algorithm to compute the average of the 10 runs
df_avg = df.groupby(['map', 'algorithm']).mean().reset_index()

# Pivot for side-by-side comparison
pivot_df = df_avg.pivot(index='map', columns='algorithm')
print("--- Mean Baseline Values ---")
display(pivot_df.round(6))

# %% [markdown]
# ### Percentage Difference Calculation
# $$\text{Percentage Difference} = \frac{\text{DHPA*} - \text{A*}}{\text{A*}} \times 100$$

# %%
results = pd.DataFrame(index=pivot_df.index)
metrics = ['search_time', 'max_memory', 'path_length']

for metric in metrics:
    a_val = pivot_df[(metric, 'a')]
    dhpa_val = pivot_df[(metric, 'dhpa')]
    
    results[f'{metric}_pct_diff'] = ((dhpa_val - a_val) / a_val) * 100

# Rename columns for clarity
results = results.rename(columns={
    'search_time_pct_diff': 'Search Phase Time Diff (%)',
    'max_memory_pct_diff': 'Max Memory Diff (%)',
    'path_length_pct_diff': 'Path Length Diff (%)'
})

print("\n--- Final Percentage Differences (DHPA* vs A*) ---")
display(results.round(2))

--- Aggregated Mean Values ---
          search_time           max_memory           path_length        
algorithm           a      dhpa          a      dhpa           a    dhpa
map                                                                     
maze         0.001417  0.000889   4.522266  4.510156      1997.0  2037.0
mp2          0.042408  0.001336  10.747266  3.298437      2871.0  2871.0
mp4          0.086642  0.001781  17.306250  3.737891      4261.0  4271.0


--- Percentage Difference (DHPA* vs A*) ---


,Search Phase Time Diff (%),Max Memory Diff (%),Path Length Diff (%)
map,,,
maze,-37.26,-0.27,2.00
mp2,-96.85,-69.31,0.00
mp4,-97.94,-78.40,0.23
